# LoRA Fine-Tuning Mistral-7B for Engineering Domain Q&A

Fine-tunes `Mistral-7B-Instruct-v0.2` using **QLoRA** (LoRA + 4-bit quantization)
on a combined dataset of research paper excerpts and synthetic Q&A pairs covering
railroad AI, DAS signal processing, and WAAM manufacturing.

**GPU:** A100 40GB (Colab Pro) — training takes ~2 hours  
**Trainable params:** ~4.2M (0.06% of 7B total)  
**Author:** Md Arifur Rahman | github.com/arifme071

**Reference papers:**
- Rahman et al., Elsevier GEITS 2024. DOI: 10.1016/j.geits.2024.100178
- Rahman et al., SPIE JARS 2024. DOI: 10.1117/1.JRS.18.016512

In [ ]:
!pip install transformers peft bitsandbytes accelerate datasets trl -q

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load Q&A dataset ──────────────────────────────────────────────────────────
# Clone the repo or load from HF Hub
!git clone https://github.com/arifme071/llm-finetuning-engineering-domain.git 2>/dev/null || echo 'Already cloned'

with open("llm-finetuning-engineering-domain/data/synthetic/qa_pairs.json") as f:
    qa_data = json.load(f)

print(f"Loaded {len(qa_data)} Q&A pairs")

# Format as Mistral instruction template
def format_instruction(sample):
    return f"""<s>[INST] {sample['instruction']} [/INST] {sample['output']}</s>"""

formatted = [{"text": format_instruction(q)} for q in qa_data]
dataset = Dataset.from_list(formatted)
print(f"\nExample:\n{formatted[0]['text'][:200]}...")

In [ ]:
# ── Load base model with 4-bit quantization (QLoRA) ───────────────────────────
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = prepare_model_for_kbit_training(model)

total = sum(p.numel() for p in model.parameters())
print(f"Base model params: {total/1e9:.2f}B")

In [ ]:
# ── Configure LoRA ────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                          # rank — higher = more capacity, more params
    lora_alpha=32,                 # scaling factor (alpha/r = 2 is standard)
    target_modules=[               # which attention matrices to adapt
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.2f}M ({100*trainable/total:.3f}% of total)")
# Expected: ~4.2M (0.06%)

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./railroad-lora-checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    gradient_checkpointing=True,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=50,
    report_to="none",
    optim="paged_adamw_8bit",       # memory-efficient optimizer
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

trainer.train()

In [ ]:
# ── Save & push LoRA adapter to HuggingFace Hub ───────────────────────────────
# Only the LoRA adapter is saved (~30MB) — not the 7B base model

HF_REPO = "arifme071/railroad-llm-lora"

model.save_pretrained("./lora-adapter")
tokenizer.save_pretrained("./lora-adapter")

# Push to Hub
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"LoRA adapter published: https://huggingface.co/{HF_REPO}")
print("Users load this adapter on top of Mistral-7B base (not included)")

In [ ]:
# ── Inference with trained LoRA model ─────────────────────────────────────────
model.eval()

def ask(question: str, max_new_tokens: int = 256) -> str:
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True)

# Test on research questions
questions = [
    "How does the sliding window post-processing improve CNN-LSTM railroad anomaly detection?",
    "What is the difference between GRU and LSTM for DAS signal processing?",
    "Explain how the HMM-RL pipeline optimizes WAAM manufacturing.",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 60)